# 02 — CoverlessStegPort Adapters & Variance Testing

Fast CPU-only workflow using the pure hexagonal `CoverlessStegPort` (`embed` + `extract`) and optional `IndexedStegAdapter` (`encode_entry` over precomputed SAM `.h5` masks).

> **Note:** `.h5` files only cache per-image SAM masks on the dataset — embedding new messages never requires regenerating `.h5` files.

In [ ]:
from pathlib import Path
import numpy as np

from coverless_steg import (
    BBoxDistanceAdapter,
    CoverlessStegPort,
    IndexedStegAdapter,
    SAMAdapter,
    apply_gaussian_noise,
    binary_to_string,
    string_to_binary,
    test_adapter_variance,
)

DATASET_DIR = Path("dataset")
CLEAN_H5 = (
    DATASET_DIR / "SAM_outputs_clean.h5"
    if (DATASET_DIR / "SAM_outputs_clean.h5").exists()
    else DATASET_DIR / "SAM_outputs.h5"
)
SOURCES = {
    name: path
    for name, path in [
        ("clean", CLEAN_H5),
        ("noise", DATASET_DIR / "SAM_outputs_noise.h5"),
        ("compression", DATASET_DIR / "SAM_outputs_compression.h5"),
    ]
    if path.exists()
}
print("Available H5 sources:", {k: str(v) for k, v in SOURCES.items()})

## 1. Embed & Extract with `SAMAdapter`

In [ ]:
sam_adapter = SAMAdapter(CLEAN_H5, grid_size=16, tolerance=0.02, mode="count")

secret_msg = "HelloWorld"
binary_msg = string_to_binary(secret_msg)

payload = sam_adapter.embed(binary_msg)
matched_keys, window_size, num_zeros = payload
extracted_bits = sam_adapter.extract(payload)
recovered_msg = binary_to_string(extracted_bits)

print(f"Secret message : {secret_msg!r} ({len(binary_msg)} bits)")
print(f"Matched keys   : {matched_keys} (window={window_size}, pad_zeros={num_zeros})")
print(f"Recovered      : {recovered_msg!r} | exact_match={extracted_bits == binary_msg}")
assert recovered_msg == secret_msg

## 2a. Custom `CoverlessStegPort` Adapter (No Masks or `.h5` Needed)

Any coverless method can implement `CoverlessStegPort` directly with just `embed` and `extract`:

In [ ]:
class MeanIntensityAdapter(CoverlessStegPort):
    """Pure CoverlessStegPort matching 4-bit chunks to RGB quadrant brightness."""

    def __init__(self, carriers: dict[str, np.ndarray]):
        self.carriers = carriers

    @staticmethod
    def _hash_image(img: np.ndarray) -> str:
        h, w = img.shape[:2]
        quads = [img[: h // 2, : w // 2], img[: h // 2, w // 2 :], img[h // 2 :, : w // 2], img[h // 2 :, w // 2 :]]
        return "".join("1" if float(np.mean(q)) >= 128.0 else "0" for q in quads)

    def embed(self, binary_msg: str):
        if not binary_msg:
            return []
        table = {}
        for k, img in self.carriers.items():
            table.setdefault(self._hash_image(img), k)
        keys = [table.get(binary_msg[i : i + 4]) for i in range(0, len(binary_msg), 4)]
        return None if any(k is None for k in keys) else keys

    def extract(self, payload, source=None) -> str:
        carriers = self.carriers if source is None else source
        return "".join(self._hash_image(carriers[k]) for k in payload)


rng = np.random.default_rng(0)
clean_imgs = {f"img_{i:03d}": rng.integers(0, 256, (8, 8, 3), dtype=np.uint8) for i in range(128)}
noisy_imgs = {k: apply_gaussian_noise(v, variance=0.01, rng=i) for i, (k, v) in enumerate(clean_imgs.items())}

pure_adapter = MeanIntensityAdapter(clean_imgs)
pure_report = test_adapter_variance(pure_adapter, clean_imgs, {"clean": clean_imgs, "noise": noisy_imgs}, ["Hi", "Port"])
print(pure_report)

## 2b. Index-Based Adapter Over Precomputed SAM `.h5` Masks (~8 Lines)

For feature/mask-lookup methods, subclass `IndexedStegAdapter` and implement `encode_entry`:

In [ ]:
class MaskAreaAdapter(IndexedStegAdapter):
    """Encode whether consecutive SAM masks grow ('1') or shrink ('0') in area."""

    def encode_entry(self, entry) -> str:
        areas = entry.get("area")
        if areas is None or len(areas) < 2:
            return ""
        return "".join("1" if d > 0 else "0" for d in np.diff(areas))


area_adapter = MaskAreaAdapter(CLEAN_H5)
keys, win, pad = area_adapter.embed(binary_msg)
assert binary_to_string(area_adapter.extract((keys, win, pad))) == secret_msg
print(f"MaskAreaAdapter matched {len(keys)} images (window={win}) and recovered {secret_msg!r}!")

## 3. Measure Bit-Error Variance Across Clean, Noise, and Compression

In [ ]:
MESSAGES = ["HelloWorld", "Coverless", "SAM Steg!", "Lazy Dev 42", "BitTest"]
reports = {}

for label, adapter in [
    ("SAMAdapter(count)", sam_adapter),
    ("SAMAdapter(binary)", SAMAdapter(CLEAN_H5, grid_size=16, tolerance=0.02, mode="binary")),
    ("BBoxDistanceAdapter", BBoxDistanceAdapter(CLEAN_H5, threshold=300.0)),
    ("MaskAreaAdapter", area_adapter),
]:
    reports[label] = test_adapter_variance(adapter, CLEAN_H5, SOURCES, MESSAGES)
    print(f"\n=== {label} ===")
    print(reports[label])

In [ ]:
for cond, summary in reports["SAMAdapter(count)"].conditions.items():
    print(f"\n[{cond}] mean_ber={summary.mean_ber:.4f}, ber_var={summary.ber_variance:.6f}")
    for m in summary.messages:
        print(
            f"  msg={m.message!r:<14} imgs={m.images_used:<2} win={m.window_size:<2} "
            f"hamming={m.hamming_distance:<3} ber={m.ber:.4f} exact={m.exact_match}"
        )